# Institution Resolver v3 — Gemma4:e4b Hakem (Judge) Colab Notebook — TERS SIRA (3'e BÖLÜNMÜŞ İŞİN %34'ü)

Amaç: `needs_review_subset.csv` (143.039 sorgu) ÜÇ makinede paralel işleniyor
(2026-08-13 kararı):

| kim | pay | satır | orijinal konum | sıra |
|---|---|---|---|---|
| Yerel | %33 | 47.203 | 1-47.203 | A→... |
| Kaggle | %33 | 47.203 | 47.204-94.406 | A→... (orta dilim) |
| **Bu Colab notebook** | **%34** | **48.633** | **94.407-143.039** | **...→A (ters)** |

Bu notebook, `data/jobs/needs_review_colab_split.csv`'yi (orijinal dosyanın SON
48.633 satırı, TERS sıralanmış) `gemma4:e4b` ile işler.

**`colab_run_qwen_judge.ipynb`'a dokunulmadı — bu tamamen ayrı, bağımsız bir notebook.**

### Resume garantisi
`inventory-batch --resume` bayrağı, çıktı CSV'sindeki `query` metnini okuyup
zaten işlenmiş sorguları atlar. Bu notebook'un çıktısı (`gemma_judge_reversed_sonuc.csv`)
diğer iki tarafın çıktısından **ayrı bir dosya** — üç taraf birbirinin ilerlemesini
bilmiyor ve bilmesine gerek yok, çünkü her biri kendi ayrı dilimini işliyor
(çakışma yok, sınırlar kesin).

Kesinti/oturum kopması durumunda: 12. hücreyi (ara yedekleme) ÖNCE çalıştırıp
ilerlemeyi Drive'a kaydedin, sonra 11. hücreyi (`--resume` ile) tekrar çalıştırmak
kaldığı sorgudan devam eder.

### Ön koşul (yerelde, bu notebook'tan ÖNCE yapılmalı)
`data/jobs/needs_review_colab_split.csv` dosyasını Google Drive'daki
`institution_resolver_v3/jobs/` klasörüne yükleyin (eski `needs_review_subset_reversed.csv`
yerine - o dosya artık kullanılmıyor, üç dilime bölündü). Ayrıca `gemma4:e4b` modeli
herkese açık Ollama registry'sinde OLMADIĞI için, Drive'da `ollama_models`
klasörünün hazır olması gerekiyor.

## 1) Donanım & GPU Kontrolü

In [ ]:
!nvidia-smi

## 2) Google Drive Bağlama + Dizin Yolları

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = "/content/drive/MyDrive/institution_resolver_v3"
DRIVE_RAW = f"{DRIVE_ROOT}/data_raw"
DRIVE_PROCESSED = f"{DRIVE_ROOT}/data_processed"
DRIVE_JOBS = f"{DRIVE_ROOT}/jobs"
DRIVE_EVAL = f"{DRIVE_ROOT}/data_eval"
DRIVE_OLLAMA = f"{DRIVE_ROOT}/ollama_models"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"

for p in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_JOBS, DRIVE_EVAL, DRIVE_OLLAMA, DRIVE_OUTPUT):
    os.makedirs(p, exist_ok=True)

print("Drive klasörleri hazır:", DRIVE_ROOT)
print("Beklenen girdi dosyası:", f"{DRIVE_JOBS}/needs_review_colab_split.csv", "- yüklendi mi kontrol edin.")

## 3) Koda Erişim (Git Clone / Pull)

In [ ]:
REPO_DIR = "/content/institution_resolver_v3"
BRANCH = "feat/gate-asama1"

import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/mcangultekin/institution_resolver_v3.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
%cd {REPO_DIR}

## 4) Drive Sembolik Linkleri

In [ ]:
import os, shutil

def _link(name, target):
    os.makedirs(target, exist_ok=True)
    link = f"data/{name}"
    if os.path.islink(link):
        return
    if os.path.isdir(link):
        for f in os.listdir(link):
            src, dst = f"{link}/{f}", f"{target}/{f}"
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(link)
    os.symlink(target, link)

os.makedirs("data", exist_ok=True)
_link("raw", DRIVE_RAW)
_link("processed", DRIVE_PROCESSED)
_link("jobs", DRIVE_JOBS)

!ls -la data
!ls -la data/jobs/needs_review_colab_split.csv

## 5) Elasticsearch Kurulumu ve Başlatma

Hakem, gate'in ürettiği aday havuzuna (candidate list) ihtiyaç duyuyor - o yüzden judge-only bir koşuda bile ES + indeks şarttır (`resolve()` her sorguda yeniden çalışır).

In [ ]:
%%bash
set -e
ES_VERSION=8.14.0
if [ ! -d /content/es ]; then
  wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-${ES_VERSION}-linux-x86_64.tar.gz -O /content/es.tar.gz
  mkdir -p /content/es
  tar -xzf /content/es.tar.gz -C /content/es --strip-components=1
fi

grep -q '^discovery.type' /content/es/config/elasticsearch.yml || cat >> /content/es/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
xpack.ml.enabled: false
EOF

cat > /content/es/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser /content/es

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
sudo -u esuser env ES_JAVA_OPTS="-Xms4g -Xmx4g -Djava.security.policy=/content/es/config/elasticsearch.policy" \
  setsid /content/es/bin/elasticsearch < /dev/null > /content/es/es.log 2>&1 &
disown
sleep 3

In [ ]:
import time, requests

for _ in range(60):
    try:
        r = requests.get("http://localhost:9200/_cluster/health", timeout=2)
        if r.status_code == 200:
            print("Elasticsearch Sağlıklı:", r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("ES başlatılamadı, /content/es/es.log kontrol edin")

## 6) Ollama Kurulumu + gemma4:e4b (Drive'dan yerel SSD'ye)

`gemma4:e4b`, `qwen3:4b-instruct-2507`'in aksine herkese açık Ollama registry'sinde
DEĞİL - Drive'daki `ollama_models` klasöründen kopyalanması gerekiyor
(bkz. `colab_run_280k.ipynb` - aynı hazırlığı paylaşıyoruz).

In [ ]:
%%bash
set -e
apt-get update -qq && apt-get install -y -qq zstd

if ! command -v ollama &> /dev/null; then
  curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr
fi

ollama --version

In [ ]:
import os, subprocess, time, requests, shutil

MODEL_TAG = "gemma4:e4b"
LOCAL_OLLAMA = "/content/ollama_local"

if not os.path.isdir(DRIVE_OLLAMA) or not os.listdir(DRIVE_OLLAMA):
    raise RuntimeError(
        f"Drive'da gemma4:e4b bulunamadı: {DRIVE_OLLAMA}\n"
        "Bu model registry'de değil, yerelden Drive'a önceden yüklenmiş olması gerekiyor "
        "(bkz. colab_run_280k.ipynb hazırlık adımları)."
    )

if not os.path.isdir(LOCAL_OLLAMA):
    print("gemma4:e4b Drive'dan yerel SSD'ye kopyalanıyor (1-2 dk)...")
    shutil.copytree(DRIVE_OLLAMA, LOCAL_OLLAMA)
    print("Kopyalama tamamlandı!")
else:
    print("Model zaten yerel SSD'de mevcut.")

os.environ["OLLAMA_MODELS"] = LOCAL_OLLAMA
# 2026-08-12 olcumu (yerel + Colab): 4 uzeri isci sayisi cokme/guvenilirlik
# kaybina yol aciyor, workers=2 en hizli+guvenilir cikti - ayni kisit burada da gecerli.
os.environ["OLLAMA_NUM_PARALLEL"] = "2"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"  # ASLA VRAM'dan düşürme

subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "a"),
    stderr=subprocess.STDOUT,
    env=os.environ,
)

for _ in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            print("Ollama servisi başlatıldı (KEEP_ALIVE=-1, NUM_PARALLEL=2)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama başlatılamadı")

print("gemma4:e4b VRAM'a ön-yükleniyor (warm-up)...")
r = requests.post("http://localhost:11434/api/generate", json={
    "model": MODEL_TAG,
    "prompt": "Merhaba",
    "stream": False,
})
if r.status_code == 200:
    print("Model GPU belleğinde sıcak ve hazır!")
else:
    print("UYARI: Warm-up başarısız, log kontrol edin:", r.text[:200])

## 7) Python Paket Kurulumu

In [ ]:
!pip uninstall -y torchaudio torchvision 2>/dev/null || true
!pip install -q --force-reinstall -e ".[dev,embed,llm,api]"

## 8) Elasticsearch Şema Sıfırlama ve İndeksleme

In [ ]:
!python3 -m institution_resolver_v3.cli.main setup-es
!python3 -m institution_resolver_v3.cli.main index --embeddings

## 9) Tekli Sorgu Testi (Her Şey Çalışıyor mu?)

In [ ]:
!python3 -m institution_resolver_v3.cli.main judge "Milli Egitim Bakanligi" --model "gemma4:e4b"

## 10) Küçük Ölçekte Hız Testi (tam koşuya geçmeden önce)

`needs_review_subset_reversed.csv`'den ilk 30 satırı (yani orijinal dosyanın SON 30 satırı) gerçek hakem çağrısıyla test eder.

In [ ]:
INPUT_CSV = f"{DRIVE_JOBS}/needs_review_colab_split.csv"
TEST_OUTPUT = "/content/gemma_reversed_test30.csv"

import time
t0 = time.time()
!python3 -m institution_resolver_v3.cli.main inventory-batch "{INPUT_CSV}" --model "gemma4:e4b" --out "{TEST_OUTPUT}" --limit 30 --workers 2
dt = time.time() - t0
print(f"\n30 satır (hakem dahil) toplam süre: {dt:.1f} sn -> {dt/30:.2f} sn/sorgu")
print("48.633 satır (bu Colab'in payı) için kaba tahmin:", f"{48633 * dt/30 / 3600:.1f} saat")

## 11) TAM KOŞU: Ters Sıralı needs_review + gemma4:e4b Hakem

Yerel makine bu dosyanın (`needs_review_subset.csv`) BAŞINDAN, bu notebook aynı
verinin TERSİNDEN başlayarak ortada buluşuyor. `--resume` sayesinde kesinti/Colab
oturum kopmasında bu hücreyi tekrar çalıştırmak kaldığı yerden devam eder -
Drive'daki `LOCAL_OUTPUT` kopyası resume için okunur.

In [ ]:
import os, shutil

INPUT_CSV = f"{DRIVE_JOBS}/needs_review_subset_reversed.csv"
LOCAL_OUTPUT = "/content/gemma_judge_reversed_sonuc.csv"
DRIVE_FINAL_OUTPUT = f"{DRIVE_OUTPUT}/gemma_judge_reversed_sonuc.csv"
MODEL_TAG = "gemma4:e4b"

# Varsa önceden işlenen kısmı yerel diske kopyala (resume için):
if os.path.exists(DRIVE_FINAL_OUTPUT) and not os.path.exists(LOCAL_OUTPUT):
    shutil.copy(DRIVE_FINAL_OUTPUT, LOCAL_OUTPUT)

print("TERS SIRALI needs_review + gemma4:e4b hakem koşusu başlıyor...")

!python3 -m institution_resolver_v3.cli.main inventory-batch "{INPUT_CSV}" --model "{MODEL_TAG}" --out "{LOCAL_OUTPUT}" --workers 2 --resume

shutil.copy(LOCAL_OUTPUT, DRIVE_FINAL_OUTPUT)
print("İŞLEM BİTTİ! Sonuç Google Drive'a kopyalandı:", DRIVE_FINAL_OUTPUT)

## 11) TAM KOŞU: Ters Sıralı needs_review + gemma4:e4b Hakem

Yerel makine bu dosyanın (`needs_review_subset.csv`) BAŞINDAN, bu notebook aynı
verinin TERSİNDEN başlayarak ortada buluşuyor. `--resume` sayesinde kesinti/Colab
oturum kopmasında bu hücreyi tekrar çalıştırmak kaldığı yerden devam eder.

**2026-08-13 düzeltmesi:** Önceden bu hücre yalnızca EN SONDA Drive'a yedek
alıyordu - internet/oturum kesintisinde araya giren TÜM ilerleme (bir seferinde
~14.000 satır) kayboldu. Artık Kaggle'daki gibi `CHUNK` satırda bir OTOMATİK
Drive yedeği alıyor - kesintide en fazla bir `CHUNK`'lık iş kaybedilir.

In [ ]:
import os, shutil, time

INPUT_CSV = f"{DRIVE_JOBS}/needs_review_subset_reversed.csv"
LOCAL_OUTPUT = "/content/gemma_judge_reversed_sonuc.csv"
DRIVE_FINAL_OUTPUT = f"{DRIVE_OUTPUT}/gemma_judge_reversed_sonuc.csv"
MODEL_TAG = "gemma4:e4b"
CHUNK = 500  # her bu kadar YENİ satırda bir Drive'a otomatik yedekle
TOTAL_TARGET = 143039  # TAM dosya

# Varsa önceden işlenen kısmı yerel diske kopyala (resume için):
if os.path.exists(DRIVE_FINAL_OUTPUT) and not os.path.exists(LOCAL_OUTPUT):
    shutil.copy(DRIVE_FINAL_OUTPUT, LOCAL_OUTPUT)

print("TERS SIRALI needs_review + gemma4:e4b hakem koşusu başlıyor (duraklat+yedekle döngüsü)...")

while True:
    done_before = 0
    if os.path.exists(LOCAL_OUTPUT):
        with open(LOCAL_OUTPUT, newline="", encoding="utf-8") as f:
            done_before = sum(1 for _ in f) - 1

    if done_before >= TOTAL_TARGET:
        print(f"TAMAMLANDI: {done_before}/{TOTAL_TARGET}")
        break

    print(f"\n--- Bu turda hedef: +{CHUNK} satır (şu ana kadar: {done_before}/{TOTAL_TARGET}) ---")
    !python3 -m institution_resolver_v3.cli.main inventory-batch "{INPUT_CSV}" --model "{MODEL_TAG}" --out "{LOCAL_OUTPUT}" --workers 2 --resume --limit {CHUNK}

    shutil.copy(LOCAL_OUTPUT, DRIVE_FINAL_OUTPUT)
    print("Otomatik yedek alındı ->", DRIVE_FINAL_OUTPUT, time.strftime("%H:%M:%S"))

    done_after = 0
    if os.path.exists(LOCAL_OUTPUT):
        with open(LOCAL_OUTPUT, newline="", encoding="utf-8") as f:
            done_after = sum(1 for _ in f) - 1
    if done_after == done_before:
        print("UYARI: bu turda hiç yeni satır işlenmedi - döngü durduruluyor (hata olabilir, logları kontrol edin).")
        break

print("\nDöngü sona erdi (oturum süresi/kota dolmuş olabilir - normal, sonraki oturumda bu hücreyi tekrar çalıştırmak --resume ile devam eder). Son durum Drive'da yedekli.")